<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_25_vlbi_practical_entry.ipynb">9.25 VLBI 实践入口</a></li>
        <li>下一节： <a href="9_27_pipeline_weblog_qa_reprocessing.ipynb">9.27 Pipeline QA、Weblog 与再处理决策</a></li>
    </ul>
</div>


## 9.26 低频与高频特殊观测体制：频率如何改变误差、策略与诊断

第 9 章前面的实践链主要以厘米波段的常规连续谱、谱线、偏振和宽场处理为骨架。这个骨架很重要，因为它建立了 visibility inspection、flag、1GC/2GC、成像、自校准、QA、科学测量和 provenance 的通用顺序。但射电观测一旦走向低频或高频，许多默认经验会失效。低频不是“频率低一点的普通成像”，高频也不是“波长短一点的普通成像”；它们分别把不同误差源推到主导地位。

低频观测通常面对宽主波束、强天空背景、RFI、电离层相位屏、Faraday 旋转、非共面基线和方向相关主波束。高频观测则面对对流层水汽路径涨落、opacity、系统温度上升、指向误差、天线表面效率、快速相位校准和绝对通量不确定度。两端的共同点是：成像参数不能脱离观测物理单独选择，校准节奏也不能由软件默认值决定。

这一节把第 7.7 节的传播效应、第 7.8 节的 RFI、第 9.12 与 9.22 节的宽场方向相关算法、第 9.23 节的观测设计连接起来。重点不是把低频和高频列成两个百科条目，而是建立一套判断方式：先识别主导误差的频率标度，再决定观测策略、校准对象、解区间、成像算法和最终验证指标。


### 9.26.1 频率标度关系：从同一组公式看两端困难

频率首先改变几何尺度。单天线主波束和干涉阵分辨率的量级分别为

$$
\theta_{\rm PB}\sim \frac{\lambda}{D},\qquad
\theta_{\rm res}\sim \frac{\lambda}{b_{\rm max}}.
$$

低频下主波束变宽，视场中会包含更多亮源、更多方向相关主波束变化和更强非共面项；同一物理基线对应的空间频率变短，角分辨率降低。高频下主波束变窄，指向误差更容易造成显著振幅误差；同一基线对应更高空间频率，角分辨率提高，但对相位稳定性和天线表面精度更敏感。

传播效应的频率标度更直接决定校准策略。电离层群路径延迟可近似写为

$$
\Delta L_{\rm iono}\simeq \frac{40.3\,\mathrm{TEC}}{\nu^2},
$$

其中 $\nu$ 用 Hz，TEC 是视线方向电子柱密度。对应的相位项近似满足 $\phi_{\rm iono}\propto \nu^{-1}$，因此低频相位极易被电离层扰动。Faraday 旋转满足

$$
\Delta\chi = \mathrm{RM}\,\lambda^2,
$$

这使低频偏振更敏感，也更容易受到电离层 RM 和方向相关 leakage 的影响。与此相反，对流层湿延迟近似非色散；若路径误差 $\delta L$ 固定，高频相位误差

$$
\delta\phi = \frac{2\pi\delta L}{\lambda}
$$

随频率升高而增大。同样的 $0.1\,\mathrm{mm}$ 水汽路径涨落，在厘米波可能只是小相位噪声，在毫米波已经足以带来明显相干损失。

天空背景也偏向低频。银河同步辐射背景常用近似 $T_{\rm sky}\propto \nu^{-2.5}$ 描述，因此低频系统温度不只来自接收机，还来自整个天空。高频系统温度则更多受大气 opacity、天气、水汽吸收线、接收机性能和仰角影响。把这些标度放在一起，就能理解为什么中等厘米波段常显得“温和”，而低频和高频都需要专门观测体制。


![frequency regime scaling map](figures/practical_frequency_regime_scaling_map.png)

图中用示意曲线展示不同误差项随频率变化的相对重要性。低频侧电离层相位、Faraday 旋转和天空背景迅速增强；高频侧对流层相位、opacity、系统温度和指向敏感性逐渐主导。曲线不是某一台望远镜的精确性能图，而是观测策略选择的物理地图。


### 9.26.2 低频观测：宽视场、电离层、RFI 与方向相关校准

低频阵列的第一特征是视场大。一个宽主波束里常同时包含目标源、多个旁瓣中的亮源、银河背景结构、太阳或强行星干扰，以及大量随方向变化的主波束响应。非共面基线项使天空不能再被简单近似为一个二维平面；w-stacking、w-projection、faceting 或类似宽场方法从可选优化变成必要条件。若强离轴源没有被正确建模，它的旁瓣和方向相关增益误差会污染整个目标场。

电离层把低频校准进一步复杂化。电离层相位屏随时间和方向变化，同一时刻不同天区方向看到的 TEC 不同，同一方向不同时间也会快速变化。方向无关自校准只能得到一个全场平均解；当视场足够大或电离层足够活跃时，这个平均解会在某些方向改善图像，在另一些方向制造残差。方向相关校准、facet-based calibration、peeling、source demixing 和 ionospheric screen fitting 的共同目标，是让校准自由度跟随真实物理自由度增长，而不是把全场误差压成一个站点相位。

RFI 是低频观测的另一个基础限制。广播、通信、雷达、卫星和本地电子设备会在动态频谱中留下窄带线、宽带突发、扫频结构和局部团簇。低频 RFI flag 不能只看单个像素幅度，还要看时间-频率形态、基线一致性和对后续带通解的污染。过少 flag 会把非高斯污染带入校准；过多 flag 会牺牲频率覆盖、影响 RM synthesis 和宽带成像。第 7.8 节介绍的鲁棒阈值、形态学扩展和 fringe stopping 抑制，在低频实践中往往是进入成像前的第一道质量门。

低频成像还常受混淆噪声和低分辨率限制。随着波束变大，未建模弱源数量增加，图像噪声不再只由热噪声决定。深低频图像的最终限制可能来自源模型不完整、主波束模型、方向相关残差和混淆噪声，而不是继续增加积分时间就能简单降低的随机噪声。


![low frequency ionosphere widefield](figures/practical_low_frequency_ionosphere_widefield.png)

图中左侧把低频视场写成多个方向：目标源只占宽主波束的一小部分，亮离轴源和电离层相位屏都会进入校准问题。右侧给出一条典型处理重点：先做 RFI excision 和方向无关初始校准，再处理亮源 peeling/demixing，随后进入电离层和主波束的方向相关求解，最后用宽场成像算法闭合到图像验证。


### 9.26.3 高频观测：对流层相干时间、opacity、指向与通量标尺

高频观测的第一限制常是对流层相位稳定性。水汽分布在望远镜上方形成快速变化的非色散路径误差，同样的路径涨落在短波长下对应更大的相位误差。若解区间内相位 RMS 为 $\sigma_\phi$，相干振幅会近似衰减为

$$
\frac{A}{A_0}\simeq \exp\left(-\frac{\sigma_\phi^2}{2}\right).
$$

这说明高频相位校准不是为了让图像更漂亮，而是为了保住可检测信号本身。快速切换、近邻相位校准源、水汽辐射计修正、self-cal、短积分时间、低 PWV 观测窗口和合适仰角范围，都是围绕相干时间展开的策略。

高频还把系统温度和 opacity 推到中心。大气透明度随频率、天气和仰角变化；水汽吸收线附近的系统温度会显著升高，低仰角时气柱加长也会提高 $T_{\rm sys}$。因此，高频数据的 QA 必须读懂 $T_{\rm sys}$、opacity、PWV、仰角、天气日志和 pipeline QA，而不能只在最终图像里估计 RMS。某些频段中，几个坏天气扫描足以决定一条谱线翼或弱连续谱结构是否可信。

指向、focus 和天线表面精度也是高频特有的实践问题。主波束随 $\lambda/D$ 缩小，几角秒指向误差可能造成可测振幅偏差；面板误差通过 Ruze 效率削弱高频增益；focus 漂移会改变主波束和耦合效率。对紧凑源，这些误差表现为时间相关振幅起伏；对 mosaic，它们还会转化为空间相关的主波束误差。高频振幅自校准若缺少可靠模型，容易把天气、指向和源结构混在一起，因此绝对通量标尺往往比中频观测更难。


![high frequency phase cadence](figures/practical_high_frequency_phase_cadence.png)

左图用同一组水汽路径 RMS 估计不同频率的相干损失：22 GHz 几乎不受影响时，230 GHz 已经出现显著相干下降。右上图示意高频快切换节奏，右下图示意系统温度随仰角和 opacity 变化。高频观测的关键元数据不只是 visibility，还包括天气、PWV、$T_{\rm sys}$、WVR、指向和 focus 日志。


### 9.26.4 观测设计：低频和高频都从科学量倒推

低频与高频的共同原则，是先把科学目标翻译成物理约束，再选择观测和处理策略。低频若目标是弥散同步辐射，需要同时考虑短基线、混淆噪声、绝对通量标尺和银河背景；若目标是低频偏振，需要考虑 $\lambda^2$ 覆盖、电离层 RM、频率分辨率和离轴 leakage；若目标是暂现源，需要考虑宽视场实时成像、RFI 占空比和去色散。高频若目标是尘埃连续谱，需要关注大气窗口、表面亮度灵敏度、相位稳定性和通量标尺；若目标是分子谱线，需要关注频率设置、边带、bandpass、线外通道和 $T_{\rm sys}$ 变化；若目标是高分辨率成像，需要关注校准源距离、fast switching 周期和自校准可行性。

低频观测通常更重视频率分辨率和方向分辨的校准自由度。过粗频率平均会损失 RFI 识别能力、加重带宽 smearing，并模糊 Faraday 结构；过长时间平均会加重时间 smearing，也会把快速电离层变化混入同一解区间。高频观测则更重视时间分辨率和相位校准节奏。过长积分会退相干，过慢切换会使校准源相位不能代表目标源，过低仰角会放大 opacity 和相位扰动。

观测设计的输出不应只是频率和总时长，而应包括一组与后处理直接相连的约束：可接受的 RFI 占空比、方向相关校准需求、最大 averaging 时间和通道宽度、校准源角距离、相位解区间、$T_{\rm sys}$ 质量门、天气选择、主波束模型需求和最终误差预算。这样，第 9.23 节中的观测设计才能真正落到低频和高频数据上。


### 9.26.5 诊断图：在成像之前判断主导失败模式

特殊频段的数据不能等到最终图像完成后才判断质量。低频诊断应从动态频谱、flag fraction、按通道/时间的占空比、校准解随方向变化、亮源残差、源位置偏移场、偏振 leakage 和 facet 间通量一致性开始。若不同方向的源位置系统漂移，常提示电离层或方向相关校准残差；若亮离轴源附近有条纹或环状残差，常提示 beam 模型、peeling 或 w-term 处理不足；若源表通量随视场位置系统变化，主波束模型和 flux scale 应先被检查。

高频诊断应从 $T_{\rm sys}$、opacity、PWV、仰角、相位校准源 RMS、WVR 修正前后比较、bandpass 稳定性、校准源结构、pointing/focus 日志和振幅随时间变化开始。若相位 RMS 在某些扫描突然升高，后续图像动态范围下降并不一定来自 CLEAN 参数；若 $T_{\rm sys}$ 在某个 spectral window 异常升高，谱线噪声和通量不确定度必须按频率分段评估；若振幅随仰角系统变化，gain curve、opacity 或 pointing 往往比自校准阈值更值得检查。


![special regime quality diagnostics](figures/practical_special_regime_quality_diagnostics.png)

图中给出两类频段的典型 QA 线索。低频侧，动态频谱中的 RFI 形态和视场内源位置偏移可直接指向 RFI 与电离层问题；高频侧，相位校准源 RMS 与 $T_{\rm sys}$ 频率结构能在成像前揭示相干损失和大气窗口问题。特殊频段的图像质量控制必须从这些物理诊断开始。


### 9.26.6 与真实软件工作流的对应

低频生态常见工具包括 AOFlagger、DP3、WSClean、DDFacet、killMS、LOFAR 相关 pipeline、CASA、CARTA 和 Python/Astropy 生态。AOFlagger 或类似工具承担时频 RFI 识别；DP3 常用于低频预处理、平均、校准和方向相关链路；WSClean 提供高性能宽场、多尺度、多频率成像；DDFacet/killMS 面向方向相关成像和校准；CARTA 适合检查 cube、残差、局部噪声、偏振和区域测量。低频项目的 provenance 应记录 RFI 策略、averaging、beam 模型、facet 划分、方向相关解、peeling 源列表和最终源表验证。

高频生态常见工作流以望远镜 pipeline、CASA 和 observatory QA 产品为核心。ALMA 这类阵列会提供 $T_{\rm sys}$、WVR、bandpass、gain、flux、flag、calibration weblog 和 imaging products；手动再处理时，关键不只是重跑 `tclean`，而是判断天气、WVR、相位校准节奏、spectral window、通量标尺和 flag 是否满足科学目标。对 VLA 高频、NOEMA、SMA 或 VLBI 高频观测，名称和文件格式会不同，但物理对象相似：opacity、相位 RMS、快切换、校准源距离、指向和系统温度必须进入误差预算。

| 频段 | 主导风险 | 常见处理重点 | 核心验证 |
|---|---|---|---|
| 低频 | 电离层、RFI、宽场 DDE、混淆 | RFI 形态学 flag、peeling、facet/DDE 校准、widefield imaging | 源位置偏移、facet 残差、通量随方向变化、偏振泄漏 |
| 高频 | 水汽相位、opacity、$T_{\rm sys}$、指向 | fast switching、WVR/天气 QA、短解区间、opacity 修正、谨慎自校准 | 相位 RMS、相干损失、$T_{\rm sys}$ 频率结构、通量标尺 |

这个对应关系也说明，低频和高频都不适合只靠默认 pipeline 图像做科学解释。pipeline 产品可以是很好的起点，但特殊频段的科学可信度来自对主导误差的独立验证。


![low high observing decision workflow](figures/practical_low_high_observing_decision_workflow.png)

图中把低频、高频和共同设计问题放在同一张决策图里。低频侧强调宽视场、电离层、RFI、方向相关校准和混淆；高频侧强调相干时间、opacity、指向、快速相位转移和通量标尺。中间列提醒，所有选择最终都应回到科学角尺度、校准源几何、时间/频率分辨率、QA 和误差预算。


### 9.26.7 教学案例：150 MHz 宽场图像与 100 GHz 分子线观测

一个低频教学案例可以设定为 $150\,\mathrm{MHz}$ 宽场连续谱成像。科学目标是测量一个星系团弥散 halo 的低频谱指数，同时场内有一个强离轴射电星系。观测处理应先用动态频谱和 flag fraction 判断 RFI 占空比，再做方向无关初始校准；若强离轴源残差主导图像，应进入 peeling 或方向相关求解；成像时需要使用宽场算法并保留足够小的通道和时间采样，避免 smearing 与电离层变化混在一起。最终结果不能只报告 halo 的积分通量，还应报告注入源恢复、源位置偏移、方向相关残差、主波束模型限制和低频通量标尺不确定度。

一个高频教学案例可以设定为 $100\,\mathrm{GHz}$ 附近的分子线和连续谱观测。科学目标是测量星形成核区中的分子气体速度结构和尘埃连续谱。观测设计应选择大气窗口较好、PWV 较低的时段，安排足够近的相位校准源，并设置可覆盖线外通道的 spectral window。处理时先检查 $T_{\rm sys}$、bandpass、WVR 修正和相位 RMS，再决定是否需要 flag 坏天气扫描或缩短 gaincal 解区间；成像后要比较不同通道噪声、moment mask 稳定性、通量标尺和 beam 变化。若某些线翼只出现在高 $T_{\rm sys}$ 通道或坏天气扫描中，物理解释必须非常保守。

这两个案例的共同训练价值在于，它们都要求从图像退回到观测条件和校准诊断。低频的关键问题常是“这个结构是否来自天空，还是来自方向相关残差”；高频的关键问题常是“这个弱信号是否在相干性和大气标尺内仍可信”。只要能回答这两个问题，特殊频段观测就不再只是复杂软件流程，而成为可审查的测量链。


### 9.26.8 本节结论

低频和高频观测把频率标度关系直接写进数据处理。低频侧，宽视场、电离层、RFI、Faraday 旋转、混淆和方向相关主波束决定校准和成像策略；高频侧，对流层水汽、opacity、系统温度、指向、focus 和快速相位校准决定相干性和通量标尺。两者都要求在成像前建立物理诊断，而不是在最终图像里用默认参数反复试探。

把这一节放在 VLBI 之后，可以形成第 9 章后半部分的完整实践拓展：9.23 解释观测设计，9.24 解释软件生态和 provenance，9.25 解释极长基线相干检测，9.26 解释频段两端的特殊误差体制。后续继续加厚时，可加入真实低频 pipeline 日志、高频 observatory weblog、天气与 $T_{\rm sys}$ QA 表、方向相关残差案例和特殊频段的误差预算模板。
